In [67]:
import pandas as pd
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [68]:
df= pd.read_csv('loan_data.csv')
df.head()

,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1


In [69]:
x=df.drop(columns=['loan_status'])
y=df.loan_status

In [70]:
xtrain,xtest,ytrain,ytest=train_test_split(x,y,train_size=0.8,random_state=42)

In [71]:
obj_cols=x.select_dtypes(include='object').columns
obj_cols

Index(['person_gender', 'person_education', 'person_home_ownership',
       'loan_intent', 'previous_loan_defaults_on_file'],
      dtype='object')

In [72]:
xtrain[obj_cols].nunique()

person_gender                     2
person_education                  5
person_home_ownership             4
loan_intent                       6
previous_loan_defaults_on_file    2
dtype: int64

In [78]:
order=df['person_education'].unique()
order

array(['Master', 'High School', 'Bachelor', 'Associate', 'Doctorate'],
      dtype=object)

In [74]:
preprocessing=ColumnTransformer(
    transformers=[
        ('oh_encoder', OneHotEncoder(sparse_output=False, handle_unknown='ignore'),
          obj_cols.drop('person_education')),
        ('ord_encoder', OrdinalEncoder(categories=order, handle_unknown='use_encoded_value', unknown_value=-1),['person_education'])
    ],
    remainder='passthrough'
)

In [75]:
main_pipeline=Pipeline(
    steps=[
        ('preprocessing',preprocessing),
        ('model', DecisionTreeClassifier(random_state=42))
    ]
)

In [82]:
grid_search_cv= GridSearchCV(
    estimator=main_pipeline,
    param_grid={
        'model__criterion' : ['gini', 'entropy', 'log_loss'],
        'model__splitter': ['best', 'random'] ,
        'model__max_depth': [None,5,10,15,50,100],
        'model__min_samples_split': [2,5,7,10],
        'model__min_samples_leaf': [1,3,5,7,10],
        'model__max_features':[None,'auto', 'sqrt', 'log2']
    }
)
grid_search_cv.fit(xtrain, ytrain)

KeyboardInterrupt: 